# N107 · 区间DP与最后一步决策

**目标：** 通过选择区间内最后发生的操作分离子问题。

**先修：** N098, N031, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** dp[l][r]；区间长度顺序；边界哨兵；最后操作；合并可行性。

## 从问题到算法

区间DP常通过“最后一步”恢复子问题独立性。戳气球若先选第一个戳的气球，邻接关系会变化；选最后戳的气球，两侧边界固定。合并石头还要记录要剩几堆，并检查每次合并减少k−1堆的整除条件。三角剖分选与边(l,r)相邻的最后三角形。

## 最小实现

**本章接口：** `burst_balloons(nums)`、`merge_stones(stones, k)`、`min_score_triangulation(values)`

In [1]:
from functools import cache
from itertools import accumulate

def burst_balloons(nums):
    a=[1]+list(nums)+[1]; n=len(a); dp=[[0]*n for _ in range(n)]
    for gap in range(2,n):
        for l in range(n-gap):
            r=l+gap; dp[l][r]=max(dp[l][k]+dp[k][r]+a[l]*a[k]*a[r] for k in range(l+1,r))
    return dp[0][-1]

def merge_stones(stones,k):
    if k<2: raise ValueError('k must be at least 2')
    n=len(stones)
    if not n: return 0
    if (n-1)%(k-1): return -1
    prefix=[0]+list(accumulate(stones))
    @cache
    def f(l,r,piles):
        length=r-l
        if length<piles or (length-piles)%(k-1): return float('inf')
        if length==1: return 0 if piles==1 else float('inf')
        if piles==1: return f(l,r,k)+prefix[r]-prefix[l]
        return min((f(l,m,1)+f(m,r,piles-1) for m in range(l+1,r,k-1)),default=float('inf'))
    return f(0,n,1)

def min_score_triangulation(values):
    n=len(values)
    if n<3: return 0
    dp=[[0]*n for _ in range(n)]
    for gap in range(2,n):
        for l in range(n-gap):
            r=l+gap; dp[l][r]=min(dp[l][k]+dp[k][r]+values[l]*values[k]*values[r] for k in range(l+1,r))
    return dp[0][-1]

## 正确性、前提与复杂度

固定最后一步后，两侧操作互不改变边界，因此可以组合各自最优解。石头区间压成p堆必须满足长度与p模k−1同余；从k堆合成1堆才加整个区间和，不能在每次分割时重复加。三角形将凸多边形分成两个独立子多边形。

**代价：** 气球和三角剖分O(n³)时间O(n²)空间。石头记忆化使用(l,r,p)状态，保守O(n³k)时间O(n²k)空间上界；递归深度O(n)，仅用于题目小规模区间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,5,8]; rows=[]
for k,x in enumerate(a):
    rows.append((k,x))
show_table(['候选最后戳下标','值'],rows)
print('完整区间最优',burst_balloons(a))
# 可直接检查一、二气球区间的独立答案。
show_table(['小输入','最优得分'],[(s,burst_balloons(s)) for s in [[],[3],[3,1],[1,5],[5,8]]])

候选最后戳下标,值
0,3
1,1
2,5
3,8


完整区间最优 167


小输入,最优得分
[],0
[3],3
"[3, 1]",6
"[1, 5]",10
"[5, 8]",48


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert burst_balloons([3,1,5,8])==167
assert merge_stones([3,2,4,1],2)==20 and merge_stones([3,2,4,1],3)==-1
assert min_score_triangulation([1,3,1,4,1,5])==13
@cache
def pop_ref(a):
    return max((a[i]*(a[i-1] if i else 1)*(a[i+1] if i+1<len(a) else 1)+pop_ref(a[:i]+a[i+1:]) for i in range(len(a))),default=0)
@cache
def merge_ref(a,k):
    if len(a)==1: return 0
    return min((sum(a[i:i+k])+merge_ref(a[:i]+(sum(a[i:i+k]),)+a[i+k:],k) for i in range(len(a)-k+1)),default=float('inf'))
from itertools import product
for a in product(range(1,4),repeat=5):
    assert burst_balloons(a)==pop_ref(a)
    for k in (2,3): assert merge_stones(a,k)==merge_ref(a,k)
print('N107: 所有本章断言通过')

N107: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释先选第一个气球难以分离子问题。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 推导合并到一堆的余数条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 312. Burst Balloons（canonical）
- 1000. Minimum Cost to Merge Stones（canonical）：[官方入口](https://leetcode.com/problems/minimum-cost-to-merge-stones/)
- 1039. Minimum Score Triangulation of Polygon（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。